# LAB 03 — Word Representations and Embeddings

Notebook chạy co-occurrence, Skip-gram, similarity, analogy và semantic search trên corpus C4 của Lab01. Các bảng bên dưới được sinh từ lần chạy notebook; không có số liệu được ghi cứng.

## 1. Imports và cấu hình

In [1]:
from pathlib import Path
import bisect
import csv
import gzip
import json
import random
import re
import time
import unittest
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.sparse import issparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity

cwd = Path.cwd().resolve()
LAB_DIR = cwd if (cwd / "cooccurrence.py").exists() else cwd / "Lab03"
if not (LAB_DIR / "cooccurrence.py").exists():
    raise FileNotFoundError("Không tìm thấy Lab03/cooccurrence.py trong thư mục hiện tại hoặc thư mục con.")
import sys
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from cooccurrence import (
    build_cooccurrence_matrix,
    build_vocabulary,
    cosine_similarity,
    iter_tokenized_sentences,
    most_similar,
    tokenize,
)

SEED = 42
MIN_COUNT = 2
EPOCHS = 10
NEGATIVE_SAMPLES = 5
MAX_TRAINING_PAIRS = 250_000
BATCH_SIZE = 2048
VECTOR_SIZE = 100
BASELINE_WINDOW = 5
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(min(8, torch.get_num_threads()))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_FILENAME = "c4-train.00000-of-01024-30K.json.gz"
RESULTS_PATH = LAB_DIR / "results.csv"
RESULT_ROWS = []

print({
    "seed": SEED,
    "python_word2vec_runtime": "PyTorch Skip-gram with negative sampling",
    "device": str(DEVICE),
    "epochs": EPOCHS,
    "negative_samples": NEGATIVE_SAMPLES,
    "max_training_pairs_per_model": MAX_TRAINING_PAIRS,
})

{'seed': 42, 'python_word2vec_runtime': 'PyTorch Skip-gram with negative sampling', 'device': 'cuda', 'epochs': 10, 'negative_samples': 5, 'max_training_pairs_per_model': 250000}


## 2. Nạp corpus và tokenization

In [2]:
data_candidates = [
    LAB_DIR.parent / "Lab01" / DATA_FILENAME,
    cwd / "Lab01" / DATA_FILENAME,
    cwd.parent / "Lab01" / DATA_FILENAME,
]
DATA_PATH = next((path.resolve() for path in data_candidates if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"Không tìm thấy corpus {DATA_FILENAME}; đã kiểm tra {data_candidates}")

records = []
with gzip.open(DATA_PATH, "rt", encoding="utf-8") as stream:
    for line_number, line in enumerate(stream, start=1):
        if not line.strip():
            continue
        record = json.loads(line)
        text = record.get("text")
        if isinstance(text, str) and text.strip():
            records.append({"document_id": f"D{line_number:05d}", "text": text})

SENTENCE_BOUNDARY_RE = re.compile(r"(?<=[.!?])\s+|[\r\n]+")
texts = [record["text"] for record in records]
document_ids = [record["document_id"] for record in records]
document_sentences = []
tokenized_sentences = []
sentence_texts = []
for text in texts:
    doc_sentences = []
    for raw_sentence in SENTENCE_BOUNDARY_RE.split(text):
        words = tokenize(raw_sentence)
        if words:
            doc_sentences.append(words)
            tokenized_sentences.append(words)
            sentence_texts.append(raw_sentence.strip())
    document_sentences.append(doc_sentences)

corpus_counts = Counter(token for sentence in tokenized_sentences for token in sentence)
TOKEN_COUNT = sum(corpus_counts.values())
WORD_VOCABULARY, INDEX_TO_WORD = build_vocabulary(tokenized_sentences, min_count=1)
corpus_summary = {
    "dataset_path": str(DATA_PATH),
    "documents": len(records),
    "sentences": len(tokenized_sentences),
    "tokens": TOKEN_COUNT,
    "vocabulary_size_min_count_1": len(WORD_VOCABULARY),
    "tokenizer": "Unicode word tokens, casefold, sentence boundaries preserved",
}
print(json.dumps(corpus_summary, ensure_ascii=False, indent=2))

{
  "dataset_path": "D:\\Study\\Semester 7\\NLP\\Lab01\\c4-train.00000-of-01024-30K.json.gz",
  "documents": 30000,
  "sentences": 635207,
  "tokens": 10970208,
  "vocabulary_size_min_count_1": 205980,
  "tokenizer": "Unicode word tokens, casefold, sentence boundaries preserved"
}


## 3. Kiểm thử core implementation và hand-calculation được cung cấp

In [3]:
class CoreImplementationTests(unittest.TestCase):
    def test_vocabulary_order_and_min_count(self):
        word_to_index, index_to_word = build_vocabulary(["dog cat dog", "cat bird"], min_count=2)
        self.assertEqual(word_to_index, {"cat": 0, "dog": 1})
        self.assertEqual(index_to_word, {0: "cat", 1: "dog"})

    def test_lab_window_one_counts(self):
        small_corpus = [
            "the cat eats fish",
            "the dog eats fish",
            "the cat likes milk",
            "the dog likes meat",
        ]
        words = ["cat", "dog", "eats", "likes", "fish", "milk", "meat"]
        vocabulary = {word: index for index, word in enumerate(words)}
        matrix = build_cooccurrence_matrix(small_corpus, vocabulary, window_size=1)
        self.assertTrue(issparse(matrix))
        self.assertEqual(matrix.shape, (7, 7))
        expected = {
            "cat": [0, 0, 1, 1, 0, 0, 0],
            "dog": [0, 0, 1, 1, 0, 0, 0],
            "eats": [1, 1, 0, 0, 2, 0, 0],
            "likes": [1, 1, 0, 0, 0, 1, 1],
        }
        for word, vector in expected.items():
            np.testing.assert_array_equal(matrix.getrow(vocabulary[word]).toarray().ravel(), vector)

    def test_cosine_parallel_vectors(self):
        self.assertAlmostEqual(cosine_similarity([1, 2, 1], [2, 4, 2]), 1.0, delta=1e-9)

    def test_zero_vector_and_shape_mismatch(self):
        self.assertEqual(cosine_similarity([0, 0], [1, 2]), 0.0)
        with self.assertRaises(ValueError):
            cosine_similarity([1, 2], [1])

    def test_most_similar_excludes_target_and_handles_oov(self):
        vocabulary = {"a": 0, "b": 1, "c": 2}
        matrix = np.asarray([[1, 0], [1, 0], [0, 1]], dtype=float)
        self.assertEqual(most_similar("a", matrix, vocabulary, top_k=2), [("b", 1.0), ("c", 0.0)])
        self.assertEqual(most_similar("missing", matrix, vocabulary), [])
        self.assertEqual(most_similar("a", matrix, vocabulary, top_k=0), [])

    def test_empty_and_sentence_boundary(self):
        empty_vocab, _ = build_vocabulary([], min_count=1)
        self.assertEqual(empty_vocab, {})
        empty_matrix = build_cooccurrence_matrix([], {}, window_size=2)
        self.assertEqual(empty_matrix.shape, (0, 0))
        vocab = {"cat": 0, "dog": 1}
        matrix = build_cooccurrence_matrix(["cat", "dog"], vocab, window_size=1)
        self.assertEqual(matrix.nnz, 0)

    def test_manual_analogy_arithmetic(self):
        result = np.asarray([8, 2, 7]) - np.asarray([5, 1, 5]) + np.asarray([5, 3, 5])
        np.testing.assert_array_equal(result, [8, 4, 7])


test_result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(CoreImplementationTests)
)
if not test_result.wasSuccessful():
    raise AssertionError("Unit tests thất bại")
print(f"CORE_TESTS: {test_result.testsRun} passed, 0 failed")

test_cosine_parallel_vectors (__main__.CoreImplementationTests.test_cosine_parallel_vectors) ... 

ok


test_empty_and_sentence_boundary (__main__.CoreImplementationTests.test_empty_and_sentence_boundary) ... 

ok


test_lab_window_one_counts (__main__.CoreImplementationTests.test_lab_window_one_counts) ... 

...

ok


test_zero_vector_and_shape_mismatch (__main__.CoreImplementationTests.test_zero_vector_and_shape_mismatch) ... 

ok


----------------------------------------------------------------------
Ran 7 tests in 0.011s

OK


CORE_TESTS: 7 passed, 0 failed


## 4. Experiment 1 — Word-context representation

In [4]:
COOCCURRENCE_PAIRS = [("doctor", "physician"), ("doctor", "hospital"), ("cat", "dog")]
cooccurrence_rows = []
for window in (1, 2, 5):
    matrix = build_cooccurrence_matrix(tokenized_sentences, WORD_VOCABULARY, window_size=window)
    total_entries = matrix.shape[0] * matrix.shape[1]
    density = float(matrix.nnz / total_entries) if total_entries else 0.0
    row = {
        "experiment": "cooccurrence",
        "window": window,
        "vocabulary_size": len(WORD_VOCABULARY),
        "matrix_shape": matrix.shape,
        "nnz": int(matrix.nnz),
        "density": density,
        "sparsity": 1.0 - density if total_entries else 0.0,
    }
    for word1, word2 in COOCCURRENCE_PAIRS:
        label = f"{word1}__{word2}"
        if word1 not in WORD_VOCABULARY or word2 not in WORD_VOCABULARY:
            similarity, status = None, "OOV"
        else:
            similarity = cosine_similarity(
                matrix.getrow(WORD_VOCABULARY[word1]),
                matrix.getrow(WORD_VOCABULARY[word2]),
            )
            status = "ok"
        row[label] = similarity
        cooccurrence_rows.append({
            "experiment": "cooccurrence_similarity", "model": "word-context counts",
            "window": window, "word1": word1, "word2": word2,
            "similarity": similarity, "status": status,
            "vocabulary_size": len(WORD_VOCABULARY), "nnz": int(matrix.nnz),
            "density": density,
        })
    RESULT_ROWS.append({
        "experiment": "cooccurrence_matrix", "model": "word-context counts",
        "window": window, "vocabulary_size": len(WORD_VOCABULARY),
        "nnz": int(matrix.nnz), "density": density, "status": "ok",
    })
    print(json.dumps(row, ensure_ascii=False))
    del matrix
    import gc
    gc.collect()

{"experiment": "cooccurrence", "window": 1, "vocabulary_size": 205980, "matrix_shape": [205980, 205980], "nnz": 5196623, "density": 0.0001224816712220332, "sparsity": 0.9998775183287779, "doctor__physician": 0.9128294995259986, "doctor__hospital": 0.6530539997844026, "cat__dog": 0.8690563980938296}


{"experiment": "cooccurrence", "window": 2, "vocabulary_size": 205980, "matrix_shape": [205980, 205980], "nnz": 9997371, "density": 0.0002356327768835048, "sparsity": 0.9997643672231165, "doctor__physician": 0.9272977922112109, "doctor__hospital": 0.7531758485013375, "cat__dog": 0.9179668372326328}


{"experiment": "cooccurrence", "window": 5, "vocabulary_size": 205980, "matrix_shape": [205980, 205980], "nnz": 20919972, "density": 0.0004930727382914136, "sparsity": 0.9995069272617085, "doctor__physician": 0.9470321485824008, "doctor__hospital": 0.8404662477819885, "cat__dog": 0.9543648080654171}


## Analysis — Co-occurrence Representation

### Số liệu ghi nhận từ output

- Vocabulary size: **205.980**; matrix shape: **205.980 × 205.980**.

| Window | nnz | Density | Sparsity | doctor–physician | doctor–hospital | cat–dog |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 5.196.623 | 0.0001224816712220332 | 0.9998775183287779 | 0.9128294995259986 | 0.6530539997844026 | 0.8690563980938296 |
| 2 | 9.997.371 | 0.0002356327768835048 | 0.9997643672231165 | 0.9272977922112109 | 0.7531758485013375 | 0.9179668372326328 |
| 5 | 20.919.972 | 0.0004930727382914136 | 0.9995069272617085 | 0.9470321485824008 | 0.8404662477819885 | 0.9543648080654171 |

Các giá trị trên được chép từ output của cell Experiment 1.
### Nhận xét theo số liệu

Vocabulary gồm **205.980 từ**, nên ma trận có **205.980 × 205.980** phần tử. Khi window tăng từ 1 lên 2 rồi 5, `nnz` tăng từ **5.196.623** lên **9.997.371** và **20.919.972**; density tăng tương ứng từ **0.0001225** lên **0.0002356** và **0.0004931**, còn sparsity giảm từ **0.9998775** xuống **0.9997644** và **0.9995069**.

Cả ba cặp đều có cosine cao hơn khi tăng window: doctor–physician là **0.9128 → 0.9273 → 0.9470**, doctor–hospital là **0.6531 → 0.7532 → 0.8405**, cat–dog là **0.8691 → 0.9180 → 0.9544**. Như vậy, trong ma trận co-occurrence của thí nghiệm này, window lớn làm các vector của ba cặp gần nhau hơn. Đây là thay đổi của score; notebook không có điểm tương đồng chuẩn do con người gán để kết luận score cao hơn đồng nghĩa với chất lượng tốt hơn.



## 5. Word2Vec baseline — Skip-gram negative sampling

Runtime hiện có PyTorch nhưng không có Gensim. Cell này triển khai Skip-gram với negative sampling (SGNS). Mỗi model lấy mẫu tối đa 100.000 cặp target-context từ toàn bộ corpus; số cặp và seed được ghi vào kết quả để tái lập.

In [5]:
W2V_WORD_TO_INDEX, W2V_INDEX_TO_WORD = build_vocabulary(tokenized_sentences, min_count=MIN_COUNT)
W2V_COUNTS = np.asarray([corpus_counts[W2V_INDEX_TO_WORD[i]] for i in range(len(W2V_INDEX_TO_WORD))], dtype=np.float64)
ENCODED_SENTENCES = [
    [W2V_WORD_TO_INDEX[token] for token in sentence if token in W2V_WORD_TO_INDEX]
    for sentence in tokenized_sentences
]
ENCODED_SENTENCES = [sentence for sentence in ENCODED_SENTENCES if len(sentence) >= 2]
SENTENCE_STARTS = []
_cursor = 0
for _sentence in ENCODED_SENTENCES:
    SENTENCE_STARTS.append(_cursor)
    _cursor += len(_sentence)
ENCODED_TOKEN_COUNT = _cursor

print({
    "documents": len(records), "raw_tokens": TOKEN_COUNT,
    "training_vocabulary_size": len(W2V_WORD_TO_INDEX),
    "min_count": MIN_COUNT, "vector_size": VECTOR_SIZE,
    "window": BASELINE_WINDOW, "epochs": EPOCHS,
    "objective": "Skip-gram with negative sampling (SGNS)", "seed": SEED,
    "device": str(DEVICE),
})

{'documents': 30000, 'raw_tokens': 10970208, 'training_vocabulary_size': 103834, 'min_count': 2, 'vector_size': 100, 'window': 5, 'epochs': 10, 'objective': 'Skip-gram with negative sampling (SGNS)', 'seed': 42, 'device': 'cuda'}


In [6]:
def sample_skipgram_pairs(window, max_pairs=MAX_TRAINING_PAIRS, seed=SEED):
    '''Sample reproducible target-context pairs across all encoded sentences.'''
    if window < 1 or ENCODED_TOKEN_COUNT == 0:
        return np.empty((0, 2), dtype=np.int64)
    rng = np.random.default_rng(seed + window)
    pairs = np.empty((max_pairs, 2), dtype=np.int64)
    filled = 0
    attempts = 0
    max_attempts = max_pairs * 20
    while filled < max_pairs and attempts < max_attempts:
        attempts += 1
        flat_position = int(rng.integers(0, ENCODED_TOKEN_COUNT))
        sentence_index = bisect.bisect_right(SENTENCE_STARTS, flat_position) - 1
        local_position = flat_position - SENTENCE_STARTS[sentence_index]
        sentence = ENCODED_SENTENCES[sentence_index]
        offsets = [offset for offset in range(-window, window + 1)
                   if offset and 0 <= local_position + offset < len(sentence)]
        if not offsets:
            continue
        offset = offsets[int(rng.integers(0, len(offsets)))]
        pairs[filled] = (sentence[local_position], sentence[local_position + offset])
        filled += 1
    if filled == 0:
        return np.empty((0, 2), dtype=np.int64)
    return pairs[:filled]


class SkipGramNegativeSampling(nn.Module):
    def __init__(self, vocabulary_size, dimension):
        super().__init__()
        self.input_vectors = nn.Embedding(vocabulary_size, dimension, sparse=True)
        self.output_vectors = nn.Embedding(vocabulary_size, dimension, sparse=True)
        nn.init.uniform_(self.input_vectors.weight, -0.5 / dimension, 0.5 / dimension)
        nn.init.zeros_(self.output_vectors.weight)


def train_word2vec(window, dimension, epochs=EPOCHS, seed=SEED, pair_seed=None):
    started = time.perf_counter()
    pair_seed = seed if pair_seed is None else pair_seed
    pairs = sample_skipgram_pairs(window=window, seed=pair_seed)
    if pairs.size == 0:
        raise ValueError("Không sinh được training pair; cần ít nhất một câu có hai token in-vocabulary.")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = SkipGramNegativeSampling(len(W2V_WORD_TO_INDEX), dimension).to(DEVICE)
    optimizer = torch.optim.SparseAdam(model.parameters(), lr=0.01)
    negative_weights = torch.as_tensor(W2V_COUNTS ** 0.75, dtype=torch.float32, device=DEVICE)
    negative_weights /= negative_weights.sum()
    negative_ids = torch.multinomial(
        negative_weights, len(pairs) * NEGATIVE_SAMPLES, replacement=True
    ).reshape(len(pairs), NEGATIVE_SAMPLES)
    rng = np.random.default_rng(seed + 10_000)
    model.train()
    epoch_losses = []
    for epoch in range(epochs):
        order = rng.permutation(len(pairs))
        epoch_loss = 0.0
        for start in range(0, len(order), BATCH_SIZE):
            batch_ids = order[start:start + BATCH_SIZE]
            batch = torch.as_tensor(pairs[batch_ids], dtype=torch.long, device=DEVICE)
            centers = batch[:, 0]
            contexts = batch[:, 1]
            negatives = negative_ids[batch_ids]
            center_vectors = model.input_vectors(centers)
            positive_vectors = model.output_vectors(contexts)
            negative_vectors = model.output_vectors(negatives)
            positive_scores = (center_vectors * positive_vectors).sum(dim=1)
            negative_scores = torch.bmm(
                negative_vectors, center_vectors.unsqueeze(2)
            ).squeeze(2)
            loss = -(
                F.logsigmoid(positive_scores).mean()
                + F.logsigmoid(-negative_scores).sum(dim=1).mean()
            )
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            epoch_loss += float(loss.detach().cpu()) * len(batch_ids)
        epoch_loss /= len(pairs)
        epoch_losses.append(epoch_loss)

    input_vectors = model.input_vectors.weight.detach().cpu().numpy().copy()
    model_size_bytes = int(
        (model.input_vectors.weight.numel() + model.output_vectors.weight.numel()) * 4
    )
    training_time = time.perf_counter() - started
    del model, optimizer, negative_ids, negative_weights
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        "vectors": input_vectors,
        "window": window,
        "dimension": dimension,
        "epochs": epochs,
        "seed": seed,
        "pair_seed": pair_seed,
        "objective": "Skip-gram + negative sampling",
        "training_pairs": len(pairs),
        "training_time": training_time,
        "model_size_bytes": model_size_bytes,
        "last_epoch_loss": epoch_loss,
        "epoch_losses": epoch_losses,
        "vocabulary_size": len(W2V_WORD_TO_INDEX),
    }


def vector_similarity(model, word1, word2):
    if word1 not in W2V_WORD_TO_INDEX or word2 not in W2V_WORD_TO_INDEX:
        return None, "OOV"
    return cosine_similarity(
        model["vectors"][W2V_WORD_TO_INDEX[word1]],
        model["vectors"][W2V_WORD_TO_INDEX[word2]],
    ), "ok"


def embedding_most_similar(model, word, top_k=5):
    if word not in W2V_WORD_TO_INDEX or top_k <= 0:
        return []
    target_index = W2V_WORD_TO_INDEX[word]
    vectors = model["vectors"]
    target = vectors[target_index]
    target_norm = np.linalg.norm(target)
    norms = np.linalg.norm(vectors, axis=1)
    denominator = norms * target_norm
    scores = np.divide(vectors @ target, denominator, out=np.zeros_like(norms), where=denominator != 0)
    ranked = [(token, float(scores[index])) for token, index in W2V_WORD_TO_INDEX.items() if index != target_index]
    ranked.sort(key=lambda item: (-item[1], item[0]))
    return ranked[:top_k]


def embedding_analogy(model, positive, negative, top_k=5):
    tokens = list(positive) + list(negative)
    if any(token not in W2V_WORD_TO_INDEX for token in tokens):
        return []
    query = np.sum([model["vectors"][W2V_WORD_TO_INDEX[token]] for token in positive], axis=0)
    query -= np.sum([model["vectors"][W2V_WORD_TO_INDEX[token]] for token in negative], axis=0)
    norms = np.linalg.norm(model["vectors"], axis=1)
    denom = norms * np.linalg.norm(query)
    scores = np.divide(model["vectors"] @ query, denom, out=np.zeros_like(norms), where=denom != 0)
    excluded = {W2V_WORD_TO_INDEX[token] for token in tokens}
    ranked = [(token, float(scores[index])) for token, index in W2V_WORD_TO_INDEX.items() if index not in excluded]
    ranked.sort(key=lambda item: (-item[1], item[0]))
    return ranked[:top_k]


MODEL_CACHE = {}
def get_model(window, dimension):
    key = (window, dimension)
    if key not in MODEL_CACHE:
        model = train_word2vec(
            window=window, dimension=dimension, epochs=EPOCHS,
            seed=SEED + window * 100 + dimension,
            pair_seed=SEED + window * 100,
        )
        MODEL_CACHE[key] = model
        RESULT_ROWS.append({
            "experiment": "word2vec_training", "model": model["objective"],
            "window": window, "dimension": dimension,
            "training_time": model["training_time"],
            "model_size_bytes": model["model_size_bytes"],
            "vocabulary_size": model["vocabulary_size"],
            "training_pairs": model["training_pairs"],
            "epochs": model["epochs"], "seed": model["seed"],
            "pair_seed": model["pair_seed"],
            "status": "ok",
        })
        print({k: v for k, v in model.items() if k != "vectors"})
    return MODEL_CACHE[key]


baseline_model = get_model(BASELINE_WINDOW, VECTOR_SIZE)
print("Baseline config:", {
    "documents": len(records), "vocabulary_size": baseline_model["vocabulary_size"],
    "vector_size": VECTOR_SIZE, "window": BASELINE_WINDOW,
    "min_count": MIN_COUNT, "epochs": EPOCHS,
    "objective": baseline_model["objective"], "seed": baseline_model["seed"],
})

{'window': 5, 'dimension': 100, 'epochs': 10, 'seed': 642, 'pair_seed': 542, 'objective': 'Skip-gram + negative sampling', 'training_pairs': 250000, 'training_time': 15.283159999991767, 'model_size_bytes': 83067200, 'last_epoch_loss': 0.9336964209365844, 'epoch_losses': [3.4618938652496336, 2.762660048553467, 2.3303302481689454, 1.922690760131836, 1.5813411259307861, 1.322934862335205, 1.148060502883911, 1.03787669090271, 0.9720052939987183, 0.9336964209365844], 'vocabulary_size': 103834}
Baseline config: {'documents': 30000, 'vocabulary_size': 103834, 'vector_size': 100, 'window': 5, 'min_count': 2, 'epochs': 10, 'objective': 'Skip-gram + negative sampling', 'seed': 642}


## Analysis — Word2Vec Baseline

### Cấu hình và quy mô dữ liệu trong output

| Thuộc tính | Giá trị |
|---|---:|
| Corpus | 30.000 documents; 10.970.208 tokens |
| Vocabulary | 103.834 (`min_count=2`) |
| Vector dimension | 100 |
| Window | 5 |
| Epochs | 10 |
| Objective | Skip-gram with negative sampling (SGNS) |
| Training pairs | 250.000 |
| Training time | 15.283159999991767 giây |
| Model size | 83.067.200 bytes |
### Vì sao các neighbor này gần `doctor`?

Top-5 là `delicate` (0.6781), `platter` (0.6618), `roadside` (0.6285), `hoarseness` (0.6271) và `intentional` (0.6241). Tần suất trong corpus lần lượt là **167, 40, 41, 4 và 69**, trong khi `doctor` xuất hiện **772** lần.

Các common contexts đầu bảng chủ yếu là từ phổ biến như `and`, `the`, `a`, `of`, `to`. Ví dụ, `doctor`–`delicate` cùng có `and` 176/79 lần, `the` 326/75 lần và `a` 313/54 lần (số thứ nhất là count của `doctor`). Câu corpus về `doctor` nói đến bác sĩ trong đào tạo hoặc điều trị; ví dụ của `delicate` nói về sương buổi sáng, âm nhạc và kiểu tóc; ví dụ của `platter` nói về đồ ăn/bàn ăn; ví dụ của `roadside` nói về motel và phong cảnh ven đường. Các câu này không cho thấy một liên hệ nội dung rõ ràng giữa những từ đó với `doctor`.

Riêng `hoarseness`, các ví dụ có nội dung về triệu chứng, GP và rối loạn thần kinh, nên có liên hệ chủ đề sức khỏe với bác sĩ. Tuy vậy, từ này chỉ xuất hiện **4** lần và các common contexts chung trong cửa sổ 5 từ rất ít (ví dụ `and` và `of` mỗi từ có count 2 ở phía `hoarseness`). Với các neighbor còn lại, bằng chứng hiện có chưa giải thích được cosine cao; **available corpus evidence is insufficient to establish a clear reason**. Những common contexts phổ biến cho thấy có thể có nhiễu từ ngữ cảnh chung, nhưng chưa đủ để khẳng định đó là nguyên nhân.



## 6. Inspect embeddings và corpus evidence

In [7]:
inspection_words = ["doctor", "hospital", "patient", "disease", "computer", "football", "banana"]
doctor_neighbors = []
for target_word in inspection_words:
    if target_word not in W2V_WORD_TO_INDEX:
        print({"target_word": target_word, "status": "OOV"})
        RESULT_ROWS.append({"experiment": "neighbors", "model": "baseline", "target_word": target_word, "status": "OOV"})
        continue
    neighbors = embedding_most_similar(baseline_model, target_word, top_k=5)
    print({"target_word": target_word, "neighbors": neighbors})
    for rank, (neighbor, score) in enumerate(neighbors, start=1):
        RESULT_ROWS.append({
            "experiment": "neighbors", "model": "baseline", "target_word": target_word,
            "rank": rank, "neighbor": neighbor, "similarity": score,
            "status": "ok",
        })
        if target_word == "doctor":
            doctor_neighbors.append((neighbor, score))


def inspect_shared_contexts(word1, word2, window=5, top_k=10, max_examples=3):
    '''Return token counts, common contexts, and exact corpus sentence examples.'''
    context_counts = {word1: Counter(), word2: Counter()}
    examples = {word1: [], word2: []}
    for sentence, raw_text in zip(tokenized_sentences, sentence_texts):
        for word in (word1, word2):
            positions = [i for i, token in enumerate(sentence) if token == word]
            if positions and len(examples[word]) < max_examples:
                examples[word].append(raw_text)
            for position in positions:
                left, right = max(0, position - window), min(len(sentence), position + window + 1)
                context_counts[word].update(
                    token for i, token in enumerate(sentence[left:right], start=left)
                    if i != position
                )
    common = sorted(
        ((token, context_counts[word1][token], context_counts[word2][token])
         for token in context_counts[word1].keys() & context_counts[word2].keys()),
        key=lambda row: (-min(row[1], row[2]), -(row[1] + row[2]), row[0]),
    )[:top_k]
    return {
        "word1": word1, "word2": word2,
        "frequency_word1": int(corpus_counts[word1]),
        "frequency_word2": int(corpus_counts[word2]),
        "common_contexts": common,
        "sentences_word1": examples[word1],
        "sentences_word2": examples[word2],
    }


doctor_evidence = [inspect_shared_contexts("doctor", neighbor) for neighbor, _ in doctor_neighbors]
for evidence in doctor_evidence:
    print(json.dumps(evidence, ensure_ascii=False))
    RESULT_ROWS.append({
        "experiment": "corpus_evidence", "model": "baseline",
        "target_word": evidence["word1"], "neighbor": evidence["word2"],
        "target_frequency": evidence["frequency_word1"],
        "neighbor_frequency": evidence["frequency_word2"],
        "shared_contexts": json.dumps(evidence["common_contexts"], ensure_ascii=False),
        "example_sentences": json.dumps(
            {"doctor": evidence["sentences_word1"], evidence["word2"]: evidence["sentences_word2"]},
            ensure_ascii=False,
        ), "status": "ok",
    })

{'target_word': 'doctor', 'neighbors': [('delicate', 0.6780673265457153), ('platter', 0.6618244051933289), ('roadside', 0.6284955739974976), ('hoarseness', 0.6270899176597595), ('intentional', 0.6240653991699219)]}

{'target_word': 'hospital', 'neighbors': [('underpins', 0.5176847577095032), ('morty', 0.5165001153945923), ('moreau', 0.5147965550422668), ('profession', 0.5134398341178894), ('crisco', 0.5093521475791931)]}


{'target_word': 'patient', 'neighbors': [('ordinarily', 0.5421642661094666), ('elective', 0.5215246081352234), ('1m', 0.5178984999656677), ('promising', 0.5088114142417908), ('collects', 0.5050267577171326)]}


{'target_word': 'disease', 'neighbors': [('gels', 0.5607876777648926), ('economist', 0.5547693967819214), ('redshift', 0.5476884245872498), ('wt', 0.5418058037757874), ('grocer', 0.538519561290741)]}


{'target_word': 'computer', 'neighbors': [('cancel', 0.5992387533187866), ('anyplace', 0.5861889123916626), ('ammo', 0.5805555582046509), ('consequential', 0.5790003538131714), ('illumination', 0.5780808925628662)]}
{'target_word': 'football', 'neighbors': [('pawn', 0.5733173489570618), ('voicemail', 0.5706067681312561), ('wires', 0.5541101098060608), ('contraction', 0.553046703338623), ('zhong', 0.551706850528717)]}


{'target_word': 'banana', 'neighbors': [('unbelievable', 0.7129169702529907), ('vain', 0.7087714672088623), ('stiffening', 0.7059574723243713), ('docked', 0.7048389315605164), ('graphed', 0.6994783878326416)]}


{"word1": "doctor", "word2": "delicate", "frequency_word1": 772, "frequency_word2": 167, "common_contexts": [["and", 176, 79], ["the", 326, 75], ["a", 313, 54], ["of", 123, 37], ["to", 252, 31], ["with", 87, 26], ["is", 80, 23], ["in", 77, 22], ["are", 34, 16], ["for", 83, 15]], "sentences_word1": ["Inspector Beck, H Division, deposed that, having sent for the doctor, he gave orders to prevent any persons leaving the court, and he directed officers to make a search.", "Assuming leadership roles at all stages of their training and career is a progressively vital component of the definition of a good doctor.", "Should your skincare “clinic”, not have an overseeing doctor, chances are that you are not receiving the appropriate numbing cream (only a 2% Lidocaine), which can make this treatment uncomfortable."], "sentences_word2": ["The pieces are so pretty and organic, like some of Nature’s most delicate moments; like morning dew.I can look forever at the tiny dainty morning drops on leave

## Số liệu output — `doctor` và corpus evidence

### Top-5 neighbors

| Rank | Từ | Similarity |
|---:|---|---:|
| 1 | delicate | 0.6780673265457153 |
| 2 | platter | 0.6618244051933289 |
| 3 | roadside | 0.6284955739974976 |
| 4 | hoarseness | 0.6270899176597595 |
| 5 | intentional | 0.6240653991699219 |

### Tần suất và common contexts trong corpus

| Candidate | Frequency | Các common contexts đầu ra |
|---|---:|---|
| doctor | 772 | — |
| delicate | 167 | and (79), the (75), a (54), of (37), to (31) |
| platter | 40 | a (23), the (15), and (13), on (13), to (7) |
| roadside | 41 | the (18), a (14), of (11), and (10), in (7) |
| hoarseness | 4 | and (2), of (2), in (2), the (1), a (1) |
| intentional | 69 | of (26), the (25), and (24), to (23), or (13) |

Cell evidence cũng in ra tối đa ba câu corpus cho `doctor` và từng candidate. Ví dụ đầu ra cho `doctor`: “Assuming leadership roles at all stages of their training and career is a progressively vital component of the definition of a good doctor.”

Ví dụ đầu ra cho các candidate: `delicate` — “The pieces are so pretty and organic, like some of Nature’s most delicate moments; like morning dew.”; `platter` — “The table is set up with a portable burner on which a cast-iron platter is placed.”; `roadside` — “And of course, you are not going to like the idea of being stranded on the terrace of a roadside motel till rescue workers pull you guys out.”; `hoarseness` — “A persistent sore throat combined with shortness of breath, problems in swallowing or an ear ache is an indication of laryngeal cancer rather than hoarseness alone, a new study warns.”; `intentional` — “Through a variety of creative opportunities at each developmental stage, Willowstone Academy is actively developing a creativity model that supports the intentional and individual understanding of creativity in each child.”



## 7. Experiment 3 — Context window

In [8]:
CONTEXT_PAIRS = [("doctor", "physician"), ("doctor", "hospital"), ("cat", "dog")]
context_window_rows = []
for window in (2, 5, 10):
    model = get_model(window, VECTOR_SIZE)
    for word1, word2 in CONTEXT_PAIRS:
        score, status = vector_similarity(model, word1, word2)
        row = {"word1": word1, "word2": word2, "window": window,
               "similarity": score, "status": status,
               "vocabulary_size": model["vocabulary_size"],
               "training_time": model["training_time"]}
        context_window_rows.append(row)
        print(row)
        RESULT_ROWS.append({"experiment": "context_window", "model": model["objective"], **row})

{'window': 2, 'dimension': 100, 'epochs': 10, 'seed': 342, 'pair_seed': 242, 'objective': 'Skip-gram + negative sampling', 'training_pairs': 250000, 'training_time': 10.608963000006042, 'model_size_bytes': 83067200, 'last_epoch_loss': 0.8910084420166016, 'epoch_losses': [3.448805700317383, 2.68585135458374, 2.2250263612213135, 1.8328122905197144, 1.5112363175811767, 1.265818026298523, 1.0973672794265747, 0.9914354402046204, 0.9287935596923829, 0.8910084420166016], 'vocabulary_size': 103834}
{'word1': 'doctor', 'word2': 'physician', 'window': 2, 'similarity': 0.27678387342092886, 'status': 'ok', 'vocabulary_size': 103834, 'training_time': 10.608963000006042}
{'word1': 'doctor', 'word2': 'hospital', 'window': 2, 'similarity': 0.1229640130677885, 'status': 'ok', 'vocabulary_size': 103834, 'training_time': 10.608963000006042}
{'word1': 'cat', 'word2': 'dog', 'window': 2, 'similarity': 0.34728964491769965, 'status': 'ok', 'vocabulary_size': 103834, 'training_time': 10.608963000006042}
{'wor

{'window': 10, 'dimension': 100, 'epochs': 10, 'seed': 1142, 'pair_seed': 1042, 'objective': 'Skip-gram + negative sampling', 'training_pairs': 250000, 'training_time': 10.780589999980293, 'model_size_bytes': 83067200, 'last_epoch_loss': 0.9418685025939941, 'epoch_losses': [3.467582057434082, 2.7780507326660158, 2.3526010902099608, 1.937323082092285, 1.5924973794555664, 1.3315086603927613, 1.1568226082458497, 1.046765799636841, 0.9802487469329834, 0.9418685025939941], 'vocabulary_size': 103834}
{'word1': 'doctor', 'word2': 'physician', 'window': 10, 'similarity': 0.11063671321359965, 'status': 'ok', 'vocabulary_size': 103834, 'training_time': 10.780589999980293}
{'word1': 'doctor', 'word2': 'hospital', 'window': 10, 'similarity': 0.028639043714585824, 'status': 'ok', 'vocabulary_size': 103834, 'training_time': 10.780589999980293}
{'word1': 'cat', 'word2': 'dog', 'window': 10, 'similarity': 0.19642474248424474, 'status': 'ok', 'vocabulary_size': 103834, 'training_time': 10.7805899999802

## Analysis — Context Window

### Similarity output

| Pair | Window 2 | Window 5 | Window 10 |
|---|---:|---:|---:|
| doctor–physician | 0.27678387342092886 | 0.4661430598395461 | 0.11063671321359965 |
| doctor–hospital | 0.1229640130677885 | 0.21824811798120858 | 0.028639043714585824 |
| cat–dog | 0.34728964491769965 | 0.2226636501929055 | 0.19642474248424474 |

Đây là các giá trị được in trong output của cell Context window.
### Trả lời câu hỏi

Từ window 2 lên 5, doctor–physician tăng **0.2768 → 0.4661** và doctor–hospital tăng **0.1230 → 0.2182**, còn cat–dog giảm **0.3473 → 0.2227**. Từ window 5 lên 10, cả ba score đều giảm: lần lượt còn **0.1106**, **0.0286** và **0.1964**. Vì vậy, window lớn hơn không phải lúc nào cũng cho score cao hơn; cat–dog đã giảm ngay ở bước 2 → 5, và cả ba cặp giảm ở bước 5 → 10.

Với cửa sổ rộng hơn, mô hình nhận thêm từ ở xa hơn trong câu, nên ngữ cảnh có thể chuyển từ lân cận trực tiếp sang chủ đề chung của câu. Trong output, window 10 làm các cặp có score thấp hơn window 5; đây là bằng chứng phù hợp với khả năng các context xa thêm nhiễu cho những cặp này. Tuy nhiên, mỗi window được huấn luyện với seed và tập cặp mẫu khác nhau (window 2: seed 342/pair seed 242; window 5: 642/542; window 10: 1142/1042), nên không thể quy toàn bộ khác biệt chỉ cho window.



## 8. Experiment 4 — Embedding dimension

In [9]:
DIMENSION_ROWS = []
for dimension in (50, 100, 300):
    model = get_model(BASELINE_WINDOW, dimension)
    sim_results = {}
    for word1, word2 in [("doctor", "physician"), ("doctor", "hospital"), ("cat", "dog")]:
        score, status = vector_similarity(model, word1, word2)
        sim_results[f"{word1}__{word2}"] = score
        RESULT_ROWS.append({
            "experiment": "dimension_similarity", "model": model["objective"],
            "window": BASELINE_WINDOW, "dimension": dimension,
            "word1": word1, "word2": word2, "similarity": score,
            "status": status,
        })
    analogy = embedding_analogy(model, ["king", "woman"], ["man"], top_k=5)
    analogy_rank = next((rank for rank, (word, _) in enumerate(analogy, start=1) if word == "queen"), None)
    row = {
        "dimension": dimension,
        "training_time_seconds": model["training_time"],
        "model_size_bytes": model["model_size_bytes"],
        "vocabulary_size": model["vocabulary_size"],
        "training_pairs": model["training_pairs"],
        "similarities": sim_results,
        "analogy_top5": analogy,
        "queen_rank_in_top5": analogy_rank,
    }
    DIMENSION_ROWS.append(row)
    print(json.dumps(row, ensure_ascii=False))
    RESULT_ROWS.append({
        "experiment": "embedding_dimension", "model": model["objective"],
        "window": BASELINE_WINDOW, "dimension": dimension,
        "training_time": model["training_time"],
        "model_size_bytes": model["model_size_bytes"],
        "vocabulary_size": model["vocabulary_size"],
        "training_pairs": model["training_pairs"],
        "analogy_rank": analogy_rank, "status": "ok",
    })

{'window': 5, 'dimension': 50, 'epochs': 10, 'seed': 592, 'pair_seed': 542, 'objective': 'Skip-gram + negative sampling', 'training_pairs': 250000, 'training_time': 10.41145419998793, 'model_size_bytes': 41533600, 'last_epoch_loss': 1.1911599567718505, 'epoch_losses': [3.5366524106903077, 2.854442798187256, 2.5156523287353516, 2.2137784179229736, 1.9478989623565675, 1.7254717577209473, 1.540683187812805, 1.3930684076690674, 1.278452759590149, 1.1911599567718505], 'vocabulary_size': 103834}
{"dimension": 50, "training_time_seconds": 10.41145419998793, "model_size_bytes": 41533600, "vocabulary_size": 103834, "training_pairs": 250000, "similarities": {"doctor__physician": 0.4870036156367077, "doctor__hospital": 0.28864153636267187, "cat__dog": 0.34934567261944954}, "analogy_top5": [["grey", 0.6517695188522339], ["sections", 0.6234334707260132], ["sheep", 0.5435773134231567], ["reception", 0.5430092811584473], ["більше", 0.5415552258491516]], "queen_rank_in_top5": null}


{"dimension": 100, "training_time_seconds": 15.283159999991767, "model_size_bytes": 83067200, "vocabulary_size": 103834, "training_pairs": 250000, "similarities": {"doctor__physician": 0.4661430598395461, "doctor__hospital": 0.21824811798120858, "cat__dog": 0.2226636501929055}, "analogy_top5": [["antiguan", 0.4632636308670044], ["kimbel", 0.4252157211303711], ["htc", 0.4171248972415924], ["materialize", 0.40250253677368164], ["jamiroquai", 0.40043631196022034]], "queen_rank_in_top5": null}


{'window': 5, 'dimension': 300, 'epochs': 10, 'seed': 842, 'pair_seed': 542, 'objective': 'Skip-gram + negative sampling', 'training_pairs': 250000, 'training_time': 14.024817700003041, 'model_size_bytes': 249201600, 'last_epoch_loss': 0.9156819717330933, 'epoch_losses': [3.38006566506958, 2.585171428375244, 1.8910318960494996, 1.379438327583313, 1.1205850839233398, 1.0087003781814574, 0.9609439276313781, 0.9372069074325562, 0.9235822295761108, 0.9156819717330933], 'vocabulary_size': 103834}


{"dimension": 300, "training_time_seconds": 14.024817700003041, "model_size_bytes": 249201600, "vocabulary_size": 103834, "training_pairs": 250000, "similarities": {"doctor__physician": 0.3763128472726957, "doctor__hospital": 0.19812445592059036, "cat__dog": 0.10047569310049616}, "analogy_top5": [["repeated", 0.39421141147613525], ["warranties", 0.38666754961013794], ["accrued", 0.38289520144462585], ["performances", 0.3767082989215851], ["tuned", 0.3693999648094177]], "queen_rank_in_top5": null}


## Analysis — Embedding Dimension

### Số liệu ghi nhận từ output

| Dimension | Training time (s) | Model size (bytes) | doctor–physician | doctor–hospital | cat–dog | `queen` trong Top-5 analogy |
|---:|---:|---:|---:|---:|---:|---|
| 50 | 10.41145419998793 | 41.533.600 | 0.4870036156367077 | 0.28864153636267187 | 0.34934567261944954 | Không |
| 100 | 15.283159999991767 | 83.067.200 | 0.4661430598395461 | 0.21824811798120858 | 0.2226636501929055 | Không |
| 300 | 14.024817700003041 | 249.201.600 | 0.3763128472726957 | 0.19812445592059036 | 0.10047569310049616 | Không |
### Nhận xét theo output

Training time tăng từ **10.41 giây** ở dimension 50 lên **15.28 giây** ở dimension 100, sau đó còn **14.02 giây** ở dimension 300; thời gian không tăng đều theo dimension. Ngược lại, model size tăng từ **41.533.600** lên **83.067.200** rồi **249.201.600 bytes**.

Trong ba cặp được đo, doctor–physician giảm **0.4870 → 0.4661 → 0.3763**, doctor–hospital giảm **0.2886 → 0.2182 → 0.1981**, cat–dog giảm **0.3493 → 0.2227 → 0.1005**. `queen` không có trong Top-5 của analogy ở cả ba dimension. Các kết quả này không cho thấy score hoặc kết quả analogy cải thiện khi tăng dimension. Notebook không có benchmark có ground truth để chấm chất lượng embedding tổng thể, nên chưa thể kết luận dimension nào là tốt nhất.



## 9. Word similarity evaluation

In [10]:
SIMILARITY_PAIRS = [
    ("doctor", "physician"), ("car", "automobile"), ("king", "queen"),
    ("cat", "dog"), ("computer", "banana"),
]
similarity_rows = []
for word1, word2 in SIMILARITY_PAIRS:
    score, status = vector_similarity(baseline_model, word1, word2)
    row = {"word1": word1, "word2": word2, "similarity": score, "status": status}
    similarity_rows.append(row)
    RESULT_ROWS.append({"experiment": "word_similarity", "model": baseline_model["objective"], **row})
ranked_similarity = sorted(
    (row for row in similarity_rows if row["similarity"] is not None),
    key=lambda row: (-row["similarity"], row["word1"], row["word2"]),
)
print("Scores:", json.dumps(similarity_rows, ensure_ascii=False))
print("Ranked descending:", json.dumps(ranked_similarity, ensure_ascii=False))

Scores: [{"word1": "doctor", "word2": "physician", "similarity": 0.4661430598395461, "status": "ok"}, {"word1": "car", "word2": "automobile", "similarity": 0.2128623283683046, "status": "ok"}, {"word1": "king", "word2": "queen", "similarity": 0.12760195009816078, "status": "ok"}, {"word1": "cat", "word2": "dog", "similarity": 0.2226636501929055, "status": "ok"}, {"word1": "computer", "word2": "banana", "similarity": 0.26736303451998383, "status": "ok"}]
Ranked descending: [{"word1": "doctor", "word2": "physician", "similarity": 0.4661430598395461, "status": "ok"}, {"word1": "computer", "word2": "banana", "similarity": 0.26736303451998383, "status": "ok"}, {"word1": "cat", "word2": "dog", "similarity": 0.2226636501929055, "status": "ok"}, {"word1": "car", "word2": "automobile", "similarity": 0.2128623283683046, "status": "ok"}, {"word1": "king", "word2": "queen", "similarity": 0.12760195009816078, "status": "ok"}]


## Analysis — Word Similarity

### Similarity output và thứ tự được in trong notebook

| Thứ tự | Pair | Similarity |
|---:|---|---:|
| 1 | doctor–physician | 0.4661430598395461 |
| 2 | computer–banana | 0.26736303451998383 |
| 3 | cat–dog | 0.2226636501929055 |
| 4 | car–automobile | 0.2128623283683046 |
| 5 | king–queen | 0.12760195009816078 |
### Expected và observed

| Cặp từ | Dự kiến ban đầu | Observed | Mức nhất quán với trực giác |
|---|---|---:|---|
| doctor–physician | Gần nhau | 0.4661, hạng 1/5 | Có |
| car–automobile | Gần nhau | 0.2129, hạng 4/5 | Một phần |
| king–queen | Gần nhau | 0.1276, hạng 5/5 | Không trong lần chạy này |
| cat–dog | Có liên hệ | 0.2227, hạng 3/5 | Một phần |
| computer–banana | Không liên quan | 0.2674, hạng 2/5 | Không |

Các trường hợp đáng chú ý là `computer`–`banana` có score cao hơn `cat`–`dog` và `car`–`automobile`, còn `king`–`queen` ở cuối danh sách. Trong corpus, `computer` có frequency **1.776**, `banana` **116**; các common contexts đầu gồm `the` (655/59), `and` (545/53), `a` (488/35). Câu ví dụ về `computer` nói đến kỹ năng máy tính/VR, còn câu ví dụ về `banana` nói đến bánh mì chuối và thực phẩm; các ví dụ này chưa cho thấy một chủ đề chung rõ ràng để giải thích score.

`king` có frequency **857**, `queen` **424**. Ví dụ corpus dùng `king` trong cụm “king prawns”, kích cỡ giường “King/Super King” và tên riêng “Troy King”; ví dụ `queen` chủ yếu nói về Queen Elizabeth. Common context `size` xuất hiện 61 lần với `king` và 56 lần với `queen`. Nhiều cách dùng khác nhau có thể làm các vector trộn nhiều ngữ cảnh; đây là một khả năng phù hợp với các ví dụ, không phải nguyên nhân được thí nghiệm kiểm chứng. Các score chỉ là cosine của model này, không phải điểm human similarity.



## 10. Word analogy

In [11]:
for positive, negative, label in [
    (["king", "woman"], ["man"], "king - man + woman"),
    (["Paris", "Italy"], ["France"], "Paris - France + Italy"),
]:
    positive = [word.casefold() for word in positive]
    negative = [word.casefold() for word in negative]
    results = embedding_analogy(baseline_model, positive, negative, top_k=5)
    row = {"analogy": label, "positive": positive, "negative": negative, "top5": results,
           "status": "ok" if results else "OOV"}
    print(json.dumps(row, ensure_ascii=False))
    for rank, (neighbor, score) in enumerate(results, start=1):
        RESULT_ROWS.append({
            "experiment": "analogy", "model": baseline_model["objective"],
            "target_word": label, "rank": rank, "neighbor": neighbor,
            "similarity": score, "status": "ok",
        })

{"analogy": "king - man + woman", "positive": ["king", "woman"], "negative": ["man"], "top5": [["antiguan", 0.4632636308670044], ["kimbel", 0.4252157211303711], ["htc", 0.4171248972415924], ["materialize", 0.40250253677368164], ["jamiroquai", 0.40043631196022034]], "status": "ok"}

{"analogy": "Paris - France + Italy", "positive": ["paris", "italy"], "negative": ["france"], "top5": [["utilities", 0.5326177477836609], ["extrinsic", 0.4976195693016052], ["blu", 0.49455878138542175], ["disc", 0.4945186376571655], ["cubs", 0.49319133162498474]], "status": "ok"}


## Analysis — Word Analogy

### `king - man + woman`: Top-5 output

| Rank | Từ | Score |
|---:|---|---:|
| 1 | antiguan | 0.4632636308670044 |
| 2 | kimbel | 0.4252157211303711 |
| 3 | htc | 0.4171248972415924 |
| 4 | materialize | 0.40250253677368164 |
| 5 | jamiroquai | 0.40043631196022034 |

`queen` không xuất hiện trong danh sách Top-5 được in ở output.
### Trả lời câu hỏi

Với `king - man + woman`, `queen` **không xuất hiện trong Top-5**, nên rank của `queen` ngoài phạm vi danh sách được in. Top-5 thực tế là `antiguan` (0.4633), `kimbel` (0.4252), `htc` (0.4171), `materialize` (0.4025) và `jamiroquai` (0.4004).

Phép tính vector này tạo một truy vấn theo hướng quan hệ hình học trong embedding space; ở lần chạy này, truy vấn không đưa từ được kỳ vọng `queen` vào Top-5. Kết quả không chứng minh mô hình hiểu quan hệ giới tính hay hoàng gia như con người. Trong corpus, `king` xuất hiện 857 lần và `queen` 424 lần; các ví dụ `king` có cả nghĩa tôm, kích cỡ giường và tên riêng, còn ví dụ `queen` có nhiều câu về Queen Elizabeth. Các tần suất/câu này cho thấy cách dùng trong corpus đa dạng, nhưng không đủ để xác định riêng nguyên nhân analogy thất bại.



## 11. Semantic search — `medical treatment`

In [12]:
QUERY = "medical treatment"
TOP_K = 5

def average_embedding(tokens, model):
    ids = [W2V_WORD_TO_INDEX[token] for token in tokens if token in W2V_WORD_TO_INDEX]
    if not ids:
        return None
    return model["vectors"][ids].mean(axis=0)


query_vector = average_embedding(tokenize(QUERY), baseline_model)
semantic_search_rows = []
if query_vector is None:
    print({"query": QUERY, "status": "OOV"})
else:
    document_vectors = np.zeros((len(records), VECTOR_SIZE), dtype=np.float32)
    document_has_vector = np.zeros(len(records), dtype=bool)
    for index, doc_sentences in enumerate(document_sentences):
        tokens = [token for sentence in doc_sentences for token in sentence]
        vector = average_embedding(tokens, baseline_model)
        if vector is not None:
            document_vectors[index] = vector
            document_has_vector[index] = True
    query_norm = np.linalg.norm(query_vector)
    doc_norms = np.linalg.norm(document_vectors, axis=1)
    semantic_scores = np.divide(
        document_vectors @ query_vector, doc_norms * query_norm,
        out=np.zeros(len(records), dtype=np.float32),
        where=(doc_norms * query_norm) != 0,
    )
    semantic_scores[~document_has_vector] = -1.0
    for rank, index in enumerate(np.argsort(-semantic_scores)[:TOP_K], start=1):
        row = {
            "rank": rank, "document_id": document_ids[index],
            "similarity": float(semantic_scores[index]),
            "preview": texts[index][:280].replace("\n", " "),
        }
        semantic_search_rows.append(row)
        print(json.dumps(row, ensure_ascii=False))
        RESULT_ROWS.append({
            "experiment": "semantic_search", "model": "Word2Vec averaged document vectors",
            "query": QUERY, "rank": rank, "document_id": document_ids[index],
            "score": float(semantic_scores[index]), "document_preview": row["preview"],
            "status": "ok",
        })


tfidf_vectorizer = TfidfVectorizer(
    tokenizer=tokenize, token_pattern=None, lowercase=False, min_df=2,
    dtype=np.float32,
)
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
tfidf_query = tfidf_vectorizer.transform([QUERY])
tfidf_scores = (tfidf_matrix @ tfidf_query.T).toarray().ravel()
tfidf_top = np.argsort(-tfidf_scores)[:TOP_K]
for rank, index in enumerate(tfidf_top, start=1):
    row = {
        "rank": rank, "document_id": document_ids[index],
        "score": float(tfidf_scores[index]),
        "preview": texts[index][:280].replace("\n", " "),
    }
    print("TF-IDF", json.dumps(row, ensure_ascii=False))
    RESULT_ROWS.append({
        "experiment": "semantic_search", "model": "TF-IDF baseline",
        "query": QUERY, "rank": rank, "document_id": document_ids[index],
        "score": float(tfidf_scores[index]), "document_preview": row["preview"],
        "status": "ok",
    })

{"rank": 1, "document_id": "D01827", "similarity": 0.578822910785675, "preview": "Stefi has degrees in medical history (UC Berkeley) and international public health (UCLA). In 2010 she co-founded the Sustainable Healthcare Education network to integrate sustainability and health into medical and nursing education. She led public health teaching at Norwich Medi"}
{"rank": 2, "document_id": "D16605", "similarity": 0.5752599239349365, "preview": "Paul continued the attack saying state law should trump federal laws. Both Bush and Christie said they support medical marijuana but not recreational use. Maria La France of Iowans for Medical Cannabis warns against not viewing the two as very different uses."}
{"rank": 3, "document_id": "D19256", "similarity": 0.573829174041748, "preview": "Welcome to the official Orlando Magic shop! Copyright © 2019 Magic Jerseys Shop. All rights reserved."}
{"rank": 4, "document_id": "D23622", "similarity": 0.5733164548873901, "preview": "Not all ultrasound sy

TF-IDF {"rank": 1, "document_id": "D08371", "score": 0.4544254541397095, "preview": "What is a Online Medical Second Opinion? For over 25 years, patients and families have looked to We Care India for medical advice and treatment. Now you may consult with our world-renowned physicians from the comfort of your home utilizing the customized second opinion service th"}
TF-IDF {"rank": 2, "document_id": "D22382", "score": 0.38027435541152954, "preview": "Hi, my name is .... well... Summer. That's not my real name, of course (names changed to protect the innocent) but I am a real person who is really on DC Workers' Compensation and receiving medical treatment. You may be wondering, is it really as difficult and challenging as they"}
TF-IDF {"rank": 3, "document_id": "D27782", "score": 0.36002346873283386, "preview": "❶Press Officer Resume Sample. Based on your requirements and skills you can shuffle these sections in the resume. Consults reference books and material including Internet to ver

## Analysis — Semantic Search

Query: `medical treatment`; mỗi danh sách dưới đây được chép từ output Top-5.

### Word2Vec

| Rank | Document ID | Similarity | Preview trong output |
|---:|---|---:|---|
| 1 | D01827 | 0.578822910785675 | Stefi has degrees in medical history (UC Berkeley) and international public health (UCLA). In 2010 she co-founded the Sustainable Healthcare Education network to integrate sustainability and health into medical and nursing education. |
| 2 | D16605 | 0.5752599239349365 | Paul continued the attack saying state law should trump federal laws. Both Bush and Christie said they support medical marijuana but not recreational use. |
| 3 | D19256 | 0.573829174041748 | Welcome to the official Orlando Magic shop! Copyright © 2019 Magic Jerseys Shop. All rights reserved. |
| 4 | D23622 | 0.5733164548873901 | Not all ultrasound systems are equal created equal. Truffles Vein Specialists has Philips Affinity ultrasound systems. |
| 5 | D05312 | 0.5692587494850159 | Decorate your walls in style with this city map print of Antibes, France! Choose your background or line color for a personalized look. |

### TF-IDF baseline

| Rank | Document ID | Score | Preview trong output |
|---:|---|---:|---|
| 1 | D08371 | 0.4544254541397095 | What is a Online Medical Second Opinion? For over 25 years, patients and families have looked to We Care India for medical advice and treatment. |
| 2 | D22382 | 0.38027435541152954 | That's not my real name, of course (names changed to protect the innocent) but I am a real person who is really on DC Workers' Compensation and receiving medical treatment. |
| 3 | D27782 | 0.36002346873283386 | Press Officer Resume Sample. Based on your requirements and skills you can shuffle these sections in the resume. |
| 4 | D12659 | 0.32927149534225464 | This guidance is for pharmacists who handle, use and sell/supply medical devices. What to do with medical device alerts. |
| 5 | D11246 | 0.31520694494247437 | Drink lots of water! HA fillers need water to settle perfectly on your skin, if you’re dehydrated you won’t get the best out of your treatment. |
### Quan sát lexical overlap và trường hợp lỗi

Toàn văn `D01827` có từ `medical` nhưng không có `treatment`; đây là kết quả hạng 1 của Word2Vec (0.5788). `D16605` nói về medical marijuana và có `medical`, nhưng không có `treatment`. `D23622` có cả “medical vascular imaging” và “treatment”. Ngược lại, toàn văn `D19256` chỉ nói về Orlando Magic shop, còn `D05312` nói về bản đồ Antibes; hai tài liệu này không có từ `medical` hay `treatment` trong nội dung đã truy hồi.

TF-IDF trả `D08371` ở hạng 1 (0.4544). Toàn văn tài liệu lặp lại trực tiếp cụm “medical” và “treatment” trong nội dung về dịch vụ medical second opinion. `D22382` ở hạng 2 cũng lặp lại “medical treatment”. Như vậy, hai phương pháp trả các tài liệu hạng 1 khác nhau; TF-IDF hạng 1 có exact query terms rõ hơn, còn danh sách Word2Vec có cả tài liệu không liên quan. Score hai phương pháp được tính trên biểu diễn khác nhau nên không so sánh trực tiếp độ lớn 0.5788 và 0.4544.



## 12. Polysemy limitation — `bank`

In [13]:
bank_examples = [
    "I deposited money in the bank.",
    "We sat on the river bank.",
]
bank_in_vocab = "bank" in W2V_WORD_TO_INDEX
if bank_in_vocab:
    bank_vector = baseline_model["vectors"][W2V_WORD_TO_INDEX["bank"]]
    print({
        "token": "bank", "vocabulary_entries": 1,
        "stored_vectors_for_token": 1,
        "vector_dimension": int(bank_vector.shape[0]),
        "vector_norm": float(np.linalg.norm(bank_vector)),
        "examples": bank_examples,
        "technical_note": "Static Word2Vec lưu một vector cho mỗi token trong vocabulary.",
    })
    RESULT_ROWS.append({"experiment": "polysemy", "model": baseline_model["objective"],
                        "target_word": "bank", "dimension": VECTOR_SIZE,
                        "status": "one static vector"})
else:
    print({"token": "bank", "status": "OOV", "examples": bank_examples})
    RESULT_ROWS.append({"experiment": "polysemy", "model": baseline_model["objective"],
                        "target_word": "bank", "status": "OOV"})

{'token': 'bank', 'vocabulary_entries': 1, 'stored_vectors_for_token': 1, 'vector_dimension': 100, 'vector_norm': 6.844221591949463, 'examples': ['I deposited money in the bank.', 'We sat on the river bank.'], 'technical_note': 'Static Word2Vec lưu một vector cho mỗi token trong vocabulary.'}


## Analysis — Static Embedding and Polysemy

### Output của implementation

| Token | Vocabulary entries | Stored vectors | Dimension |
|---|---:|---:|---:|
| `bank` | 1 | 1 | 100 |

Các câu được đưa vào cell: “I deposited money in the bank.” và “We sat on the river bank.”

Trong implementation này, hai câu dùng cùng một entry `bank` và cùng một vector 100 chiều. Vì vậy, Word2Vec tĩnh lưu một vector chung cho token `bank`, không tạo vector riêng theo occurrence cho câu gửi tiền và câu bờ sông. Vector đó gom các cách dùng xuất hiện trong corpus; đây là giới hạn biểu diễn được minh họa bởi hai câu trên.



## Error Analysis Candidates

Các cặp dưới đây được chọn từ output similarity để đối chiếu với corpus; phần giải thích chỉ nêu khả năng khi có bằng chứng kèm theo.

| Nhóm | Observed | Expected | Tần suất target/candidate | Shared contexts và corpus evidence | Ghi chú |
|---|---|---|---|---|---|
| Hợp lý: doctor–physician | 0.4661 | Hai từ chỉ người hành nghề y được kỳ vọng gần nhau | 772/265 | Common contexts: `the` 326/121, `a` 313/102, `and` 176/73. Ví dụ `physician`: “Ajayi is a board-certified family physician and maintains an active clinical practice.” | Score đứng hạng 1/5; ví dụ corpus ở cùng chủ đề y tế. |
| Hợp lý: car–automobile | 0.2129 | Hai từ chỉ ô tô được kỳ vọng gần nhau | 3.337/139 | Common contexts: `the` 1.673/69, `a` 1.094/36, `in` 685/26. Ví dụ hai từ đều xuất hiện trong câu về dịch vụ vận chuyển/sửa xe. | Score thấp hơn doctor–physician dù evidence câu có cùng chủ đề. |
| Hợp lý: cat–dog | 0.2227 | Hai từ thường có liên hệ trong chủ đề thú nuôi | 649/1.128 | Common contexts: `the` 284/429, `a` 237/483, `and` 149/330. Một câu corpus có nguyên cụm “my dog and cat are doing something cute together”. | Score hạng 3/5; câu ví dụ có cả hai từ. |
| Bất ngờ: doctor–delicate | 0.6781 | Không kỳ vọng là cặp gần về nghĩa | 772/167 | Common contexts đầu là `and` 176/79, `the` 326/75, `a` 313/54. Ví dụ `doctor` nói về chăm sóc sức khỏe; `delicate` có câu về sương buổi sáng và kiểu tóc. | Ví dụ chưa cho thấy liên hệ nội dung rõ; evidence hiện có chưa xác định được nguyên nhân score cao. |
| Bất ngờ: doctor–platter | 0.6618 | Không kỳ vọng là cặp gần về nghĩa | 772/40 | Common contexts đầu là `a` 313/23, `the` 326/15, `and` 176/13. Ví dụ `platter` nói về đĩa phục vụ thức ăn và bàn ăn. | Context chung phần lớn là từ phổ biến; corpus evidence chưa giải thích được cosine cao. |
| Bất ngờ: computer–banana | 0.2674 | Không kỳ vọng là cặp gần về nghĩa | 1.776/116 | Common contexts đầu là `the` 655/59, `and` 545/53, `a` 488/35. Ví dụ `computer` nói về kỹ năng máy tính/VR; `banana` nói về bánh mì chuối/thực phẩm. | Ví dụ chưa cho thấy chủ đề chung. Mỗi model chỉ train tối đa 250.000 cặp; đây là giới hạn thiết lập, chưa thể khẳng định là nguyên nhân. |



## Experimental Findings vs Initial Predictions

Mục này đối chiếu bốn dự đoán nêu trong prompt; không sửa `prediction.md`.

### Prediction 1

**Dự đoán:** `doctor` và `physician` gần nhau.

**Observed:** Cosine = **0.4661**, cao nhất trong năm pair của word-similarity experiment. Frequency là 772 và 265; các câu corpus về `physician` có nội dung chăm sóc y tế.

**Mức hỗ trợ:** **Có**, trong phạm vi model và các cặp được thử.

### Prediction 2

**Dự đoán:** Similarity thay đổi khi window từ 2 lên 5.

**Observed:** doctor–physician **0.2768 → 0.4661**; doctor–hospital **0.1230 → 0.2182**; cat–dog **0.3473 → 0.2227**.

**Mức hỗ trợ:** **Có**; cả ba score đều thay đổi, nhưng không cùng chiều. Mỗi window dùng seed và cặp huấn luyện khác nhau nên đây không phải phép cô lập duy nhất biến window.

### Prediction 3

**Dự đoán:** Dimension 50 → 100 → 300 không chắc cải thiện monotonic.

**Observed:** doctor–physician **0.4870 → 0.4661 → 0.3763**; doctor–hospital **0.2886 → 0.2182 → 0.1981**; cat–dog **0.3493 → 0.2227 → 0.1005**. `queen` không có trong Top-5 analogy ở cả ba dimension.

**Mức hỗ trợ:** **Có**; các score quan sát được không tăng theo dimension. Không có metric benchmark ground-truth để kết luận chất lượng tổng thể.

### Prediction 4

**Dự đoán:** Corpus nhỏ không đảm bảo `doctor` và `physician` gần nhau.

**Observed:** Corpus dùng trong notebook có 30.000 documents; score doctor–physician là **0.4661** và đứng hạng 1/5 trong word-similarity list.

**Mức hỗ trợ:** **Chưa rõ**. Một score trên một corpus không kiểm chứng được mệnh đề “không đảm bảo”; prompt cũng không định nghĩa ngưỡng score nào được xem là gần.



## Experimental Conclusions

1. **Context window làm thay đổi biểu diễn.** Với co-occurrence, ba pair đều tăng score từ window 1→2→5; ví dụ doctor–hospital **0.6531→0.7532→0.8405**. Với Word2Vec, doctor–physician tăng **0.2768→0.4661** từ window 2→5 nhưng còn **0.1106** ở window 10.
2. **Window lớn hơn không luôn tốt hơn.** Cat–dog giảm **0.3473→0.2227→0.1964** ở Word2Vec khi window 2→5→10. Các model còn dùng seed và cặp mẫu khác nhau nên mức ảnh hưởng riêng của window chưa được cô lập hoàn toàn.
3. **Dimension lớn hơn không luôn cải thiện kết quả.** Size tăng từ **41.533.600 bytes** (50 chiều) lên **249.201.600 bytes** (300 chiều); doctor–physician giảm **0.4870→0.3763**, và `queen` không xuất hiện trong Top-5 analogy ở cả ba cấu hình. Không có benchmark chuẩn để chọn model “tốt nhất”.
4. **Tần suất không giải thích similarity một mình.** `doctor` có frequency **772**, `hoarseness` chỉ **4** nhưng là neighbor với score **0.6271**. Đây là một quan sát trong model này, không chứng minh quan hệ nhân quả giữa frequency và cosine.
5. **Một số quan hệ có vị trí gần trong embedding đã học.** doctor–physician có score **0.4661** và đứng đầu năm pair; car–automobile **0.2129**, cat–dog **0.2227**. Đây là độ gần hình học của vector, không phải bằng chứng model hiểu như con người.
6. **Failure quan sát được:** `queen` vắng mặt trong Top-5 của `king - man + woman`; `doctor` có neighbor `delicate`/`platter` với score lần lượt **0.6781/0.6618** dù các ví dụ corpus không cho thấy liên hệ rõ; semantic search xếp Orlando Magic shop `D19256` ở hạng 3 và bản đồ Antibes `D05312` ở hạng 5.
7. **Static embedding gộp polysemy.** Output cho `bank` ghi một vocabulary entry, một vector 100 chiều; hai câu về ngân hàng tiền gửi và bờ sông cùng dùng vector này.



## 13. Xuất `results.csv`

In [14]:
CSV_FIELDS = [
    "experiment", "model", "window", "dimension", "word1", "word2", "similarity",
    "target_word", "rank", "neighbor", "query", "document_id", "score",
    "training_time", "status", "vocabulary_size", "nnz", "density",
    "model_size_bytes", "training_pairs", "epochs", "seed", "target_frequency",
    "neighbor_frequency", "shared_contexts", "example_sentences", "document_preview",
    "analogy_rank",
]
with RESULTS_PATH.open("w", newline="", encoding="utf-8-sig") as stream:
    writer = csv.DictWriter(stream, fieldnames=CSV_FIELDS, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(RESULT_ROWS)
print({
    "results_csv": str(RESULTS_PATH),
    "rows": len(RESULT_ROWS),
    "columns": CSV_FIELDS,
    "file_bytes": RESULTS_PATH.stat().st_size,
})

{'results_csv': 'D:\\Study\\Semester 7\\NLP\\Lab03\\results.csv', 'rows': 95, 'columns': ['experiment', 'model', 'window', 'dimension', 'word1', 'word2', 'similarity', 'target_word', 'rank', 'neighbor', 'query', 'document_id', 'score', 'training_time', 'status', 'vocabulary_size', 'nnz', 'density', 'model_size_bytes', 'training_pairs', 'epochs', 'seed', 'target_frequency', 'neighbor_frequency', 'shared_contexts', 'example_sentences', 'document_preview', 'analogy_rank'], 'file_bytes': 18139}
